# Resultados: rejilla, semillas y test

**Finalidad**: ver cómo se eligió cada red y qué salió en test, siempre junto a la persistencia.
**Flujo**: Rejilla → Mejora frente a la persistencia → Semillas → Elección → Test → Por año
**Requiere**: los runs de `runs/` y `artifacts/`. No entrena ni evalúa: lee lo guardado.
**Decisiones que ilustra**: apartados 7 y 8 de `docs/DECISIONES.md`.

Test aparece desde el apartado 5. Sus cifras describen el resultado; ninguna decisión puede apoyarse en ellas.

In [ ]:
import json

import pandas as pd

from oil_rnn import explore as ex
from oil_rnn import explore_plots as plots
from oil_rnn.config import load_config
from oil_rnn.paths import SPLITS_MANIFEST

cfg = load_config()
time_col, target = cfg["data"]["time_column"], cfg["data"]["target"]
manifest = json.loads(SPLITS_MANIFEST.read_text(encoding="utf-8"))

runs = ex.runs_table()
rejilla = runs[runs["prefix"] == "g1"]
runs.groupby("prefix").size().rename("runs").to_frame().T

## 1. Rejilla: 36 configuraciones × 3 celdas × 4 horizontes

Cada celda es un run de `make cv`: la pérdida *pinball* media de los 4 pliegues. Más claro es mejor. Filas: capas (`c`), dropout (`d`) y tasa de aprendizaje.

In [ ]:
for h in cfg["sweep"]["horizons"]:
    display(plots.grid_heatmaps(rejilla, h, "cv_pinball"))

No hay una zona claramente mejor: las diferencias entre configuraciones vecinas son pequeñas e irregulares. Es lo que se espera cuando hay poca señal.

## 2. Mejora frente a la persistencia: cuántas configuraciones la baten

`cv_skill_vs_baseline = 1 − RMSE de la red / RMSE de la persistencia`, en porcentaje. Por encima de 0, la red es mejor.

In [ ]:
for h in (5, 63):
    display(plots.grid_heatmaps(rejilla, h, "cv_skill_vs_baseline", fmt="+.1f", percent=True))

In [ ]:
resumen = rejilla.groupby("horizon").agg(
    configuraciones=("run_id", "size"),
    mejoran_rmse=("cv_skill_vs_baseline", lambda s: int((s > 0).sum())),
    mejoran_pinball=("cv_pinball_skill", lambda s: int((s > 0).sum())),
    mejor_skill=("cv_skill_vs_baseline", "max"),
    skill_mediano=("cv_skill_vs_baseline", "median"),
)
resumen.style.format({"mejor_skill": "{:+.1%}", "skill_mediano": "{:+.1%}"})

Elegir la mejor de 108 configuraciones sobre los mismos 4 bloques da una cifra optimista: la mejor siempre parece buena, aunque sea por azar (decisión 7.4).

## 3. Semillas: ¿se distinguen las tres celdas?

La mejor configuración de cada celda y horizonte, con las semillas 42, 7 y 123. Si los puntos de dos celdas se solapan, la diferencia no supera al azar de la inicialización.

In [ ]:
otras = runs[runs["prefix"].isin(["g1s7", "g1s123"])]
base = pd.concat(
    [ex.same_config(rejilla, r) for _, r in otras[otras["prefix"] == "g1s7"].iterrows()]
)
tres_semillas = pd.concat([base, otras])
display(plots.seeds_dots(tres_semillas, "cv_pinball"))
tres_semillas.groupby(["horizon", "cell"])["cv_pinball"].agg(["mean", "std", "count"]).round(3)

## 4. Elección: la regla fijada antes de mirar test

En cada horizonte, la celda con menor `cv_pinball` medio en 3 semillas. Quedó escrita en `docs/PLAN.md` antes de evaluar (decisión 7.5).

In [ ]:
medias = tres_semillas.groupby(["horizon", "cell"]).agg(
    pinball=("cv_pinball", "mean"),
    skill_rmse=("cv_skill_vs_baseline", "mean"),
    skill_pinball=("cv_pinball_skill", "mean"),
    cobertura_corregida=("cv_coverage_cqr", "mean"),
)
elegidas = medias.loc[medias.groupby("horizon")["pinball"].idxmin()]
elegidas.style.format("{:+.1%}", subset=["skill_rmse", "skill_pinball"]).format(
    "{:.0%}", subset=["cobertura_corregida"]
)

## 5. Test: una sola evaluación

Modelos finales entrenados con train + val. Cada cifra va con su referencia.

In [ ]:
finales = runs[runs["prefix"] == "final"].set_index("horizon").sort_index()
tabla = pd.DataFrame(
    {
        "red": finales["cell"].str.upper(),
        "RMSE red": finales["test_rmse"],
        "RMSE persistencia": finales["test_baseline_rmse"],
        "mejora RMSE": finales["test_skill_vs_baseline"],
        "MAE red": finales["test_mae"],
        "MAE persistencia": finales["test_baseline_mae"],
        "dirección red": finales["test_direction_accuracy"],
        "dirección referencia": finales["test_baseline_direction_accuracy"],
        "cobertura red": finales["test_coverage"],
        "cobertura referencia": finales["test_baseline_coverage"],
    }
)
pct = [
    "mejora RMSE",
    "dirección red",
    "dirección referencia",
    "cobertura red",
    "cobertura referencia",
]
tabla.style.format(
    "{:.2f}", subset=[c for c in tabla.columns if "RMSE " in c or "MAE" in c]
).format("{:+.1%}", subset=["mejora RMSE"]).format("{:.0%}", subset=pct[1:])

Ninguna red mejora a la persistencia en RMSE. A 63 filas la dirección sí acierta más que la referencia, pero la magnitud falla.

In [ ]:
for h, row in finales.iterrows():
    pred = ex.test_predictions(row["run_id"])
    display(plots.predictions_plot(pred, f"h = {h} · {row['cell'].upper()} en test"))

A 63 filas la red amplifica el movimiento reciente: predice que la subida sigue cuando el precio ya ha vuelto.

## 6. Por año: dónde falla

In [ ]:
por_año = {h: ex.skill_by(ex.test_predictions(r["run_id"])) for h, r in finales.iterrows()}
display(
    pd.DataFrame({f"h = {h}": t["skill_rmse"] for h, t in por_año.items()}).style.format("{:+.1%}")
)
display(
    pd.DataFrame({f"h = {h}": t["coverage"] for h, t in por_año.items()}).style.format("{:.0%}")
)
plots.bars_by_group(
    por_año[5], "coverage", "h = 5 · cobertura del intervalo por año (%)", reference=0.8
)

En 2026 la cobertura cae en los horizontes cortos: el precio se movió más de lo que el intervalo, calibrado con años más tranquilos, esperaba (decisión 9.2).

## 7. Resumen: lo que hay que saber decir

- **Conclusión**: en 2023–2026 ninguna red mejora a «el precio de hoy». Es un resultado, no un fallo de proceso.
- **Las tres celdas** no se distinguen: su diferencia es menor que la variación entre semillas.
- **Lo más fiable**: el intervalo a 5 filas, con cobertura cercana al 80 %, aunque no mejor que el histórico.
- **La lección**: la mejora de validación a 63 filas venía de un pliegue y de elegir entre 108 candidatos.
- **Brentcast** llega a lo mismo a 3 meses y 1 año; su mejora a 1 semana viene del futuro del Brent, no de la red (apartado 11 de `docs/DECISIONES.md`).